Carga Inicial de los Dataset Proprocionados

In [2]:
import numpy as np
import pandas as pd
import streamlit as st

df_customers_all = pd.read_csv('../../documents/customers_dataset.csv') # Correcto
df_order_items_all = pd.read_csv('../../documents/order_items_dataset.csv') # Correcto
df_order_payments_all = pd.read_csv('../../documents/order_payments_dataset.csv') # Correcto
df_order_reviews_all = pd.read_csv('../../documents/order_reviews_dataset.csv') # Correcto
df_orders_all = pd.read_csv('../../documents/orders_dataset.csv') # Correcto
df_product_category_name_translation_all = pd.read_csv('../../documents/product_category_name_translation.csv') # Correcto
df_products_all = pd.read_csv('../../documents/products_dataset.csv')
df_sellers_all = pd.read_csv('../../documents/sellers_dataset.csv') # Correcto

Ejercicio 3.1. Número de pedidos que llegan tarde por ciudad

In [75]:
df_orders_all['order_delivered_customer_date'] = pd.to_datetime(df_orders_all['order_delivered_customer_date'])
df_orders_all['order_estimated_delivery_date'] = pd.to_datetime(df_orders_all['order_estimated_delivery_date'])

days_measurement = (df_orders_all['order_delivered_customer_date'] - df_orders_all['order_estimated_delivery_date']).dt.days

df_late_orders = df_orders_all[
    (df_orders_all['order_delivered_customer_date'] > df_orders_all['order_estimated_delivery_date']) 
    & (df_orders_all['order_status'] == 'delivered') & (days_measurement > 0)
]

df_late_orders_city = pd.merge(df_late_orders, df_customers_all, on='customer_id')

df_late_orders_city.groupby('customer_city').size().sort_values(ascending=False).to_frame().reset_index().rename(columns={'customer_city' : 'Ciudad', 0 : 'Cant. Pedidos'}).head(n=25)

,Ciudad,Cant. Pedidos
0,sao paulo,715
1,rio de janeiro,706
2,salvador,174
3,belo horizonte,137
4,porto alegre,136
5,campinas,119
6,brasilia,118
7,niteroi,96
8,fortaleza,94
9,sao goncalo,83


Ejercicio 3.2. Porcentaje de pedidos retrasados respecto al total de pedidos de la ciudad

In [5]:
df_customers_orders_all = pd.merge(df_orders_all, df_customers_all, on='customer_id')

df_orders_percentage = pd.merge(df_customers_orders_all.groupby('customer_city').size().reset_index(name='total_orders'),
                                df_late_orders_city.groupby('customer_city').size().reset_index(name='total_late_orders'), on='customer_city', how='left').fillna(0)

df_orders_percentage['percentage'] = round((df_orders_percentage['total_late_orders'] / 
                                      df_orders_percentage['total_orders']) * 100, 2)

df_orders_percentage.sort_values(by=['total_late_orders'], ascending=[False]).reset_index().rename(columns={'customer_city' : 'Ciudad', 'percentage' : 'Porcentaje'}).head(n=25)

,index,Ciudad,total_orders,total_late_orders,Porcentaje
0,3597,sao paulo,15540,715.0,4.60
1,3155,rio de janeiro,6882,706.0,10.26
2,3247,salvador,1245,174.0,13.98
3,453,belo horizonte,2773,137.0,4.94
4,2964,porto alegre,1379,136.0,9.86
5,707,campinas,1444,119.0,8.24
6,558,brasilia,2131,118.0,5.54
7,2461,niteroi,849,96.0,11.31
8,1374,fortaleza,654,94.0,14.37
9,3471,sao goncalo,409,83.0,20.29


Ejercicio 3.3 Tiempo medio de retraso en días

In [ ]:
df_mean_time_days = df_late_orders_city.copy()
df_mean_time_days['order_delivered_customer_date'] = df_mean_time_days['order_delivered_customer_date'].astype('date64[pyarrow]')
df_mean_time_days['order_estimated_delivery_date'] = df_mean_time_days['order_estimated_delivery_date'].astype('date64[pyarrow]')

df_mean_time_days['late_days'] = (df_mean_time_days['order_delivered_customer_date'] - df_mean_time_days['order_estimated_delivery_date']).dt.days

df_mean_time_days.groupby('customer_city')['late_days'].mean().sort_values(ascending=False).to_frame().rename(columns={'customer_city' : 'Ciudad', 'late_days' : 'Media Dias de Retraso'})

SyntaxError: 'return' outside function (3197467089.py, line 7)

Determinar causa de insatisfacción de los clientes

In [71]:
df_late_orders_reviews = pd.merge(df_mean_time_days, df_order_reviews_all, on='order_id', how='left')

bins = [0,2,5,10,20, int(df_late_orders_reviews['late_days'].max())]
labels = ['0-2 dias', '2-5 dias', '5-10 dias', '10-20 dias', '20+ dias']

df_late_orders_reviews['range'] = pd.cut(df_late_orders_reviews['late_days'], bins=bins, labels=labels)

df_late_orders_rating_ranges = round(df_late_orders_reviews.groupby('range', observed=True)['review_score'].mean(), 2).reset_index()

df_late_orders_count_ranges = df_late_orders_reviews.groupby('range', observed=True).size().reset_index().rename(columns={ 0 : 'count'})

print(df_late_orders_rating_ranges)
print(df_late_orders_count_ranges)

        range  review_score
0    0-2 dias          3.51
1    2-5 dias          2.47
2   5-10 dias          1.77
3  10-20 dias          1.68
4    20+ dias          1.77
        range  count
0    0-2 dias   1374
1    2-5 dias   1403
2   5-10 dias   1680
3  10-20 dias   1305
4    20+ dias    800
